In [1]:
import os
import pandas as pd
import plotly.express as px

In [2]:
#CONFIG
audio_root = 'train_audio'
taxonomy_csv = 'taxonomy.csv'
output_html = 'birdclef2025_visuals.html'

In [3]:
# Count .ogg files per ID folder
ogg_counts = {}
for folder_name in os.listdir(audio_root):
    folder_path = os.path.join(audio_root, folder_name)
    if os.path.isdir(folder_path):
        ogg_files = [f for f in os.listdir(folder_path) if f.endswith('.ogg')]
        ogg_counts[folder_name] = len(ogg_files)

In [4]:
# Load taxonomy
taxonomy = pd.read_csv(taxonomy_csv)
taxonomy = taxonomy.rename(columns={taxonomy.columns[0]: 'id',
                                    taxonomy.columns[3]: 'name',
                                    taxonomy.columns[4]: 'class'})

In [5]:
# Merge counts with taxonomy
df = pd.DataFrame(list(ogg_counts.items()), columns=['id', 'ogg_count'])
df = df.merge(taxonomy[['id', 'name', 'class']], on='id', how='left')

In [6]:
# Plot 1 - OGG Count per Name
fig1 = px.bar(df.sort_values('ogg_count', ascending=False),
            x='name', y='ogg_count', title='OGG File Count per Bird Name')

In [7]:
# Plot 2 - Average OGG per Name by Class
avg_df = df.groupby('class').agg(avg_ogg_per_name=('ogg_count', 'mean')).reset_index()
fig2 = px.bar(avg_df.sort_values('avg_ogg_per_name', ascending=False),
            x='class', y='avg_ogg_per_name', title='Average OGG Files per Name by Class')

In [8]:
# Save to HTML
from plotly.subplots import make_subplots
from plotly.offline import plot

with open(output_html, 'w') as f:
    f.write(fig1.to_html(full_html=False, include_plotlyjs='cdn'))
    f.write('<hr>')
    f.write(fig2.to_html(full_html=False, include_plotlyjs=False))